In [ ]:
# Transformer Encoder-Decoder 구조 이해 - MultiHeadAttention

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, MultiHeadAttention, LayerNormalization, Dense
from tensorflow.keras.models import Model

# 하이퍼 파라미터 설정
max_len = None   # 입출력 시퀀스의 최대길이
d_model = 16    # 임베딩 차원 (각 단어가 표현되는 벡터 길이)
num_heads = 2   # 어텐션 헤드 수
dff = 32
epochs = 30
batch_size = 4

sentences = [
    "I have a pen",
    "You are nice",
    "He loves her",
    "We eat pizza"
]

In [ ]:
tokenizer = Tokenizer(filters='', lower=False, oov_token='<OOV>')   # oov : 단어사전외 단어
tokenizer.fit_on_texts(sentences)
print(tokenizer.word_index)
seqs = tokenizer.texts_to_sequences(sentences)
print(seqs)

max_len = max(len(s) for s in seqs)
encoder_input_data = pad_sequences(seqs, maxlen=max_len, padding='post')
print(encoder_input_data)

# 디코더 입력/타겟 생성 : 어휘 사전 크기와 시작 토큰 설정
vocab_size = len(tokenizer.word_index) + 1
sos_token = vocab_size   # 시작 토큰 <sos>의 정수 인덱스를 어휘 사전 마지막 번호로 지정

tokenizer.word_index['<sos>'] = sos_token
vocab_size += 1

# 디코더 입력 : [<sos>] + encoder_input_data[:,:-1]
decoder_input_data = np.zeros_like(encoder_input_data)
print(decoder_input_data)
decoder_input_data[:, 0] = sos_token
decoder_input_data[:, 1:] = encoder_input_data[:, :-1]  # 입력 문장을 한 칸씩 밀기
# 목적 : 디코더는 훈련 중, 정답 문장을 한 토큰씩 왼쪽으로 밀기한 형태를 입력으로 받아야 함

decoder_target_data = encoder_input_data  # Transformer 디코더의 학습목표(정답)를 설정


In [ ]:
# Transformer 구성
def get_transformerFunc(vocab_size, max_len):
    # 인코더 입력 : 정수 시퀀스 형태로 들어옴 (예: [2,5,3,0,0] -> "I like you" + 패딩)
    enc_inputs = Input(shape=(max_len, ))
    emb_layer = Embedding(vocab_size, d_model, mask_zero=True)
    enc_emb = emb_layer(enc_inputs)

    # Multi-head Self-Attention
    atten_out = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(enc_emb, enc_emb)
    # 잔차 연결 + Layer Normalization(학습 안정화)
    out1 = LayerNormalization(epsilon=1e-6)(enc_emb + atten_out)
    # FFN
    ffn = Dense(dff, activation='relu')(out1)
    ffn = Dense(d_model)(ffn)
    # 또 잔차 연결
    enc_out = LayerNormalization(epsilon=1e-6)(out1 + ffn)  # 최종 인코더 출력

    # 디코더 입력 : <sos> + 문장일부 : 예 : <sos> I have a -> [10, 1,2,3]처럼 정수 인코딩된 입력
    dec_inputs = Input(shape=(max_len, ))
    dec_emb = emb_layer(dec_inputs)   # 인코더와 디코더가 같은 임베딩 레이어 사용(공통된 단어사전 사용)

    # 1단계 : Self-Attention
    attn1 = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(dec_emb, dec_emb)
    out2 = LayerNormalization(epsilon=1e-6)(dec_emb + attn1)

    # 2단계 : 인코더-디코더 어텐션
    attn2 = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(out2, enc_out)
    out3 = LayerNormalization(epsilon=1e-6)(out2 + attn2)

    # 위치별 독립적으로 작동하는 2층 신경망
    ffn2 = Dense(dff, activation='relu')(out3)
    ffn2 = Dense(d_model)(ffn2)

    # 마지막으로 잔차 연결 + 정규화 -> 디코더 최종 출력
    dec_out = LayerNormalization(epsilon=1e-6)(out3 + ffn2)

    final = Dense(vocab_size, activation='softmax')(dec_out)
    return Model([enc_inputs, dec_inputs], final)   # 입력2 -> 출력 1 형태의 트렌스포머 모델

# 모델 생성 및 컴파일
transformer = get_transformerFunc(vocab_size, max_len)
transformer.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
transformer.summary()

print('Encoder input shape : ', encoder_input_data.shape)
print('Decoder input shape : ', decoder_input_data.shape)
print('Decoder target shape : ', decoder_target_data.shape)


In [ ]:
# 모델 학습
history = transformer.fit(
    [encoder_input_data, decoder_input_data],
    np.expand_dims(decoder_target_data, -1),
    epochs = epochs,
    batch_size = batch_size,
    verbose=1
)

In [ ]:
# 디코더 테스트(입력 문장 재생성)  입력문장->정수시퀀스-><sos>추가->디코딩->단어복원
def decode_transformerFunc(sentences):
    seq = tokenizer.texts_to_sequences([sentences])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    dec_in = np.zeros_like(pad)
    dec_in[0, 0] = sos_token
    dec_in[0, 1:] = pad[0, :-1]

    # 모델 예측
    pred = transformer.predict([pad, dec_in])
    tokens = np.argmax(pred[0], axis=-1)
    words = [tokenizer.index_word.get(tok, '') for tok in tokens]
    return ' '.join(words)

print('테스트 결과 ---')
for s in sentences:
    print(f'{s} => {decode_transformerFunc(s)}')

print(decode_transformerFunc('She loves pizza'))